# SafeCrime: Data Science & Machine Learning Study (2001 - 2021)
### Exploratory Data Science, Policy Intervention Testing, Risk Profiling & Predictive Modeling on Crimes Against Women in India

**Author**: SafeVision Analytics Team  
**Dataset**: National Crime Records Bureau (NCRB) longitudinal dataset (21 years across 36 States & UTs)  
**Library**: `safecrime` Python package  

---

## 📌 Research Overview & Pipeline Architecture
This notebook conducts a rigorous statistical and machine learning investigation into reported crimes against women across 36 Indian states and union territories. The study is structured across 6 core data science stages:
1. **Data Ingestion & Sanitization**: Schema validation, state name standardization, and pre-2020 baseline isolation.
2. **Distributional Statistics & Inequality**: Skewness, Kurtosis, CAGR, and spatial Gini inequality coefficients.
3. **Empirical Policy Testing**: Evaluating the statistical significance of the 2013 Nirbhaya Criminal Law Amendment Act via parametric t-tests, Mann-Whitney U tests, and Cohen's d effect sizes.
4. **Feature Engineering**: Computing the multidimensional Women Safety Risk Index (WSRI).
5. **Unsupervised Archetyping**: K-Means Clustering and 2D Principal Component Analysis (PCA).
6. **Predictive Time-Series Modeling & Anomaly Detection**: Polynomial Ridge regression forecasting with 95% confidence intervals and multivariate Isolation Forests.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path to import safecrime library
sys.path.append('..')

from safecrime import (
    CRIME_COLS,
    CRIME_LABELS,
    CrimeDataLoader,
    DataCleaner,
    RiskIndexCalculator,
    CrimeClusterModel,
    CrimeForecaster,
    CrimeAnomalyDetector,
    PolicyImpactAnalyzer,
    CrimeDistributionProfiler,
    CrimeVisualizer,
    SafeCrimePipeline
)

print("All data science and safecrime libraries loaded successfully.")

## 1. Data Ingestion & Sanitization
We load the NCRB longitudinal dataset and clean naming inconsistencies across census/state reporting changes.

In [ ]:
loader = CrimeDataLoader('../crime_data.csv')
cleaner = DataCleaner()

raw_df = loader.load()
df = cleaner.clean(raw_df)
baseline_df = cleaner.get_clean_baseline(df, max_year=2019)

print(f"Total Clean Records: {len(df):,} spanning {df['Year'].min()} - {df['Year'].max()}")
print(f"Total Crimes Recorded: {df['Total'].sum():,}")
df.head()

## 2. Longitudinal National Trends & Milestone Visualizations
Visualizing nationwide aggregated incident counts, highlighting the 2013 post-Nirbhaya reporting surge and 2020 COVID-19 mobility restriction dips.

In [ ]:
yearly_df = df.groupby('Year')[CRIME_COLS].sum().reset_index()
yearly_df['Total'] = yearly_df[CRIME_COLS].sum(axis=1)

visualizer = CrimeVisualizer()
fig_trend = visualizer.plot_national_trend(yearly_df)
plt.show()

## 3. Statistical Distribution & Spatial Gini Inequality Analysis
We calculate the Gini coefficient measuring geographic concentration inequality of crime across states, alongside category-level CAGR, Skewness, and Kurtosis.

In [ ]:
profiler = CrimeDistributionProfiler()
cat_profiles = profiler.profile_categories(yearly_df)
gini_dict = profiler.profile_state_inequality(baseline_df)

profile_df = pd.DataFrame(cat_profiles).T
profile_df['state_gini_inequality'] = [gini_dict[c] for c in profile_df.index]
profile_df

## 4. Policy Impact Testing: 2013 Nirbhaya Criminal Law Amendment
Using parametric two-sample t-tests, non-parametric Mann-Whitney U tests, and Cohen's d effect sizes to rigorously prove whether reporting surges post-2013 were statistically significant.

In [ ]:
analyzer = PolicyImpactAnalyzer()
impact_results = analyzer.analyze_nirbhaya_law_impact(baseline_df)
impact_df = pd.DataFrame(impact_results).T
impact_df[['label', 'mean_pre_2013', 'mean_post_2013', 'percentage_change', 'mann_whitney_p_value', 'cohens_d', 'statistically_significant']]

## 5. Feature Engineering: Women Safety Risk Index (WSRI)
Computing weighted crime severity scores based on Indian Penal Code injury and trauma weights, producing a normalized 0-100 metric for each State / UT.

In [ ]:
calculator = RiskIndexCalculator()
wsri_series, state_avg = calculator.compute(baseline_df)
leaderboard = pd.DataFrame(calculator.get_leaderboard(wsri_series))

plt.figure(figsize=(10, 8))
sns.barplot(data=pd.concat([leaderboard.head(7), leaderboard.tail(7)]), x='wsri', y='state', palette='flare')
plt.title('Top 7 High-Risk vs Top 7 Low-Risk States (WSRI Scale: 5 - 98)')
plt.xlabel('Women Safety Risk Index (WSRI)')
plt.show()

## 6. Unsupervised Machine Learning: K-Means Clustering & 2D PCA
Clustering states into 4 distinct structural archetypes (Critical Hotspots, Severe Domestic Abuse, Moderate Zones, Safe Havens).

In [ ]:
state_total_avg = baseline_df.groupby('State')['Total'].mean()
cluster_model = CrimeClusterModel(n_clusters=4, random_state=42)
state_clusters = cluster_model.fit_predict(state_avg, wsri_series, state_total_avg)

fig_pca = visualizer.plot_clusters_pca(state_clusters)
plt.show()

## 7. Predictive Forecasting & Anomaly Detection (2022 - 2026)
Training a regularized 2nd-degree Polynomial Ridge Regressor and multivariate Isolation Forest on longitudinal crime trajectories.

In [ ]:
forecaster = CrimeForecaster(degree=2, alpha=1.0)
forecasts, historical_fit, metrics = forecaster.fit_predict(yearly_df)

print(f"Forecaster R2 Score: {metrics['r2_score']:.4f} | RMSE: {metrics['rmse']:,.0f} cases")
fig_forecast = visualizer.plot_forecast(historical_fit, forecasts)
plt.show()

## 8. Inter-Crime Category Correlation Heatmap
Evaluating Pearson co-occurrence dynamics across offense categories.

In [ ]:
corr_matrix = baseline_df[CRIME_COLS].corr()
fig_corr = visualizer.plot_correlation_heatmap(corr_matrix)
plt.show()